Outline

This file contains a script for formatting Annual Hours of Peak Hour Excessive Delay Per Capita for use of RITIS data. The prerequisites needed for this to work, is to manually download all of the PHED files for the given UZA's, and to store all of these files into one folder. In addition, all of these files should be named using the following example schematic: Annual Hours PHED Per Capita_3-7pm_Austin_TX. The only thing the user will be changing in the file name should be the city and the state abbreviation. The PHED files can be made via the [NPMRDS analytics tool](https://npmrds.ritis.org/analytics/my-dashboard/) and selecting the MAP-21 widget. Search for your UZA, select the Annual Hours of Peak Hour Excessive Delay Per Capita box, and then add all of your years. You will be redirected to see a chart for all of the years selected detailing PHED. You can save this data, in the top right of the panel widget. A good tip to know, is you are able to simply edit your already existing widget to swap out the UZA and the name of the file, rather than re-inputting all of the years again. Additionally, to get the Percent of Eligible Miles missing PHED, you need to make each UZA's dashboard to only feature the last year with full data. So for 2024, we use 2023 data. From there, you look at the bottom right of the chart produced and subtract that number from 100%. 

***


Preparing Workspace


In [ ]:

export=True

from pathlib import Path
import os
import pandas as pd
import plotly.express as px
import sys


PATH_GIT = Path.home() / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
PATH_CODE    = PATH_GIT / 'Data' / 'RITIS'
PATH_CONFIG0 = PATH_GIT / 'config'
PATH_CONFIG  = PATH_CODE / 'config'

sys.path.append(str(PATH_CONFIG0))
import functions as func

# SharePoint
PATH_SP = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' 
PATH_CONGESTION = PATH_SP / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Congestion'
PATH_PHED  = PATH_CONGESTION / 'RITIS' / 'PHED'
PATH_LOTTR = PATH_CONGESTION / 'RITIS' / 'LOTTR'
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")




Congestion_1


In [ ]:


# Importing the data to follow a naming pattern

df_list = []

# Iterate over every file in path
for filename in os.listdir(PATH_PHED):
    if filename.endswith('.csv'):
        
        # Extract city name from the filename
        uza = filename.split('_')[2]
        
        # Load the file into a df
        file_path = PATH_PHED / filename
        df = pd.read_csv(file_path)
        
        # Adding new col w/ UZA name. This is for joining later
        df['UZA'] = uza

        df_list.append(df)

# Joining now
df_congestion1 = pd.concat(df_list, axis=0, ignore_index=True)
df_congestion1 = df_congestion1.dropna().reset_index(drop=True)
df_congestion1['Month'] = pd.to_datetime(df_congestion1['Month']) #format = "%Y/%m"
df_congestion1['Month'] = df_congestion1['Month'].dt.to_period('M')
display(df_congestion1.head())

# QC
df_plot = df_congestion1.copy()

df_plot['Year'] = df_plot['Month'].dt.year
df_plot = df_plot[df_plot['Year'] != 2011]
df_plot = df_plot.groupby(['Year', 'UZA']).sum(numeric_only = True)
df_plot = df_plot.reset_index()
df_plot = df_plot[['Year', 'UZA', 'PHED (hours)']]

fig = px.line(df_plot, x='Year', y='PHED (hours)', color='UZA', markers=True)
fig.update_layout(title = 'Total PHED (hours) by UZA')


fig.show()




Congestion_3


For Congestion_3, we only need data for SACOG counties. To do this, we use the same tool as in Congestion_1, but now we use the MPA for SACOG instead. Select the first three measures, and do the same as we did before. 

In [ ]:


# reading files
df_truck = pd.read_csv(PATH_LOTTR / 'Truck Travel Time Reliability - Sacramento.csv'             )
df_inter = pd.read_csv(PATH_LOTTR / 'Interstate Travel Time Reliability - Sacramento.csv'        )
df_noint = pd.read_csv(PATH_LOTTR / 'Non-interstate NHS Travel Time Reliability - Sacramento.csv')

# specifying LOTTR for int and non-int
df_inter.rename(columns={'LOTTR (%)': 'Interstate LOTTR (%)'    }, inplace=True)
df_noint.rename(columns={'LOTTR (%)': 'Non-Interstate LOTTR (%)'}, inplace=True)

# Merge
df_congestion3 = df_truck.merge(df_inter, on='Month', how='left').merge(df_noint, on='Month', how='left')
df_congestion3['Month'] = pd.to_datetime(df_congestion3['Month']) #format = "%Y/%m"
df_congestion3['Month'] = df_congestion3['Month'].dt.to_period('M')
df_congestion3 = df_congestion3.dropna()
df_congestion3 = df_congestion3.reset_index(drop=True)

# Get the averages per year now
display(df_congestion3)



***

Exporting

***

In [ ]:


if export:
    
    sample_type = 'RITIS'
    year_start = df_plot['Year'].min()
    year_end = df_plot['Year'].max()


    indicator = 'Congestion_1'
    geography = 'UZA'
    df_about = func.write_about(sample_type, indicator, year_start, year_end, geography=geography)
    files_out = [PATH_CONGESTION / indicator / f"{indicator} UZA RITIS.xlsx", PATH_SERVER / f"{indicator} UZA RITIS.xlsx"]

    for file_out in files_out:
        with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
            df_about      .to_excel(writer, index=False, sheet_name='About', header=False)
            df_congestion1.to_excel(writer, index=False, sheet_name='UZA'                )


    indicator = 'Congestion_3'
    geography = 'MPO'
    df_about = func.write_about(sample_type, indicator, year_start, year_end, geography=geography)
    files_out = [PATH_CONGESTION / indicator / f"{indicator} SACOG RITIS.xlsx", PATH_SERVER / f"{indicator} SACOG RITIS.xlsx"]

    for file_out in files_out:
        with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
            df_about      .to_excel(writer, index=False, sheet_name='About', header=False)
            df_congestion3.to_excel(writer, index=False, sheet_name='SACOG'              )


